# Rama hibrido sobre CIFAR-100 — HQNN hibrida (CNN + QNN) + ConvHQVAE

Version para **CIFAR-100** de `XAI/HQ_CNN_CVAE.ipynb`. Mantiene la misma estructura
en seis secciones, los mismos modelos, el mismo protocolo de entrenamiento y la
misma bateria de tecnicas de explicabilidad. La unica variable que cambia
respecto al notebook de MNIST es el **contenido de las imagenes**.

Este cuaderno es el gemelo de `Classical_CNN_CVAE.ipynb`: ambos comparten pipeline de datos,
semilla, arquitectura convolucional e hiperparametros, y difieren unicamente en
el bloque cuantico.
Cualquier diferencia observada entre los dos es por tanto atribuible al
componente cuantico y no a decisiones de diseno colaterales.

**Indice**

1. Preprocesamiento de datos y modelizacion del ruido mixto
2. Clasificacion de ruido con HQNN hibrida (CNN + QNN)
3. Denoising con ConvHQVAE
4. Explicabilidad (XAI) del clasificador
5. Explicabilidad (XAI) del modelo de denoising
6. Conclusiones y guia de comparacion con el notebook gemelo

> **Por que CIFAR-100.** En MNIST cerca del 80 % de los pixeles son fondo negro
> exacto ($x = 0$), donde el speckle ($x + x\eta$) no tiene ningun efecto
> mientras que la sal-y-pimienta resulta muy visible. Eso ofrece al clasificador
> un atajo — mirar si el fondo esta limpio — que no requiere aprender las
> estadisticas del ruido. En imagenes naturales ese atajo desaparece, lo que
> convierte a CIFAR-100 en un control de validez de los resultados obtenidos
> sobre MNIST.

## 1. Preprocesamiento de datos y modelizacion del ruido mixto

### 1.1. Imports

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset, random_split

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

from common_utils import (
    plot_training_curves,
    plot_confusion_matrix,
    evaluate_model,
    mostrar_reconstrucciones,
)

### 1.2. Configuracion del dataset

In [ ]:
# ---------------------------------------------------------------------
# CONFIGURACION DEL DATASET
# ---------------------------------------------------------------------
# Estas dos constantes son el unico punto que hay que tocar para pasar de
# la variante "CIFAR nativo" a la variante "CIFAR emparejado con MNIST".
#
#   IMG_SIZE = 32  -> resolucion nativa de CIFAR-100 (variante por defecto)
#   IMG_SIZE = 28  -> se redimensiona a 28x28, de modo que TODAS las
#                     dimensiones internas coinciden exactamente con las de
#                     los notebooks de MNIST y la unica variable que cambia
#                     entre datasets es el contenido de la imagen.
#
#   GRAYSCALE = True  -> 1 canal. Recomendado: el modelo de ruido de este TFM
#                        es por pixel y esta definido sobre imagenes en escala
#                        de grises; ademas mantiene identica la arquitectura.
#   GRAYSCALE = False -> 3 canales RGB. Requiere la mascara compartida de
#                        sal-y-pimienta (ya implementada mas abajo).
# ---------------------------------------------------------------------
IMG_SIZE = 32
GRAYSCALE = True

IN_CH = 1 if GRAYSCALE else 3

# Tras dos convoluciones stride=2 el mapa espacial queda en IMG_SIZE // 4.
FEAT = IMG_SIZE // 4

# Dimension aplanada del encoder del VAE (64 canales) y del clasificador (32).
ENC_FLAT = 64 * FEAT * FEAT
CLF_FLAT = 32 * FEAT * FEAT

print(f"Imagen      : {IN_CH}x{IMG_SIZE}x{IMG_SIZE}")
print(f"Mapa conv   : {FEAT}x{FEAT}")
print(f"Flatten VAE : {ENC_FLAT}   (MNIST 28x28 -> 3136)")
print(f"Flatten CLF : {CLF_FLAT}   (MNIST 28x28 -> 1568)")

### 1.3. Reproducibilidad (semilla)

In [ ]:
import random
import os
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seed fijada: {SEED}")

### 1.4. Carga de CIFAR-100 y submuestreo

In [ ]:
# CIFAR-100 se convierte a escala de grises (si GRAYSCALE) y se redimensiona a
# IMG_SIZE antes de pasar a tensor en [0, 1], que es el rango que asume el
# modelo de ruido.
_tf = []
if GRAYSCALE:
    _tf.append(transforms.Grayscale(num_output_channels=1))
if IMG_SIZE != 32:
    _tf.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))
_tf.append(transforms.ToTensor())

transform = transforms.Compose(_tf)

X_train_full = datasets.CIFAR100(root="./data", train=True,  download=True, transform=transform)
X_test_full  = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

In [ ]:
n_samples_train = 400
n_samples_test = 100

# En los notebooks de MNIST el submuestreo se hacia recortando `.data` y
# `.targets`. En CIFAR-100 `.data` es un ndarray HWC y `.targets` una lista,
# asi que se usa `Subset`, que es equivalente y no depende de la version de
# torchvision. Las etiquetas originales de CIFAR (las 100 categorias) no se
# utilizan: la etiqueta de este problema es el tipo de ruido mixto.
X_train = Subset(X_train_full, range(n_samples_train))
X_test  = Subset(X_test_full,  range(n_samples_test))

In [ ]:
train_size = 300
val_size = 100

split_generator = torch.Generator()
split_generator.manual_seed(SEED)

train_subset, val_subset = random_split(
    X_train,
    [train_size, val_size],
    generator=split_generator
)

### 1.5. Modelizacion del ruido mixto

Tres procesos elementales combinados por pares dan lugar a las tres clases:

- clase 0 = gaussiano + speckle
- clase 1 = gaussiano + sal y pimienta
- clase 2 = sal y pimienta + speckle

La mascara de `salt_pepper` se comparte entre canales para que el ruido
impulsivo siga siendo blanco/negro tambien en la variante RGB; con 1 canal el
comportamiento es identico al de los notebooks de MNIST.

In [ ]:
def gaussian_noise(img, sigma=0.25, generator=None):
    noise = torch.randn(img.shape, generator=generator, dtype=img.dtype) * sigma
    return torch.clamp(img + noise, 0, 1)


def salt_pepper(img, prob=0.15, generator=None):
    """Ruido impulsivo.

    A diferencia de la version de MNIST, la mascara se genera con forma
    (1, H, W) y se difunde sobre los canales. En una imagen RGB una mascara
    independiente por canal produciria pixeles de color aleatorio en lugar de
    impulsos blancos y negros, que es lo que el proceso fisico describe.
    Con 1 canal el comportamiento es identico al de los notebooks de MNIST.
    """
    noisy = img.clone()

    mask = torch.rand(
        (1, *img.shape[1:]),
        generator=generator,
        dtype=img.dtype
    ).expand_as(img)

    noisy[mask < prob / 2] = 0
    noisy[mask > 1 - prob / 2] = 1

    return noisy


def speckle(img, sigma=0.35, generator=None):
    noise = torch.randn(img.shape, generator=generator, dtype=img.dtype) * sigma
    return torch.clamp(img + img * noise, 0, 1)

In [ ]:
def apply_mixed_noise(img, generator):
    r = torch.randint(low=0, high=3, size=(1,), generator=generator).item()

    if r == 0:
        img = gaussian_noise(img, generator=generator)
        img = speckle(img, generator=generator)
        label = 0
    elif r == 1:
        img = gaussian_noise(img, generator=generator)
        img = salt_pepper(img, generator=generator)
        label = 1
    else:
        img = salt_pepper(img, generator=generator)
        img = speckle(img, generator=generator)
        label = 2

    return img, label

In [ ]:
def to_disp(t):
    """Convierte un tensor/array CHW en algo que `imshow` acepte.

    Devuelve HxW si la imagen tiene 1 canal y HxWx3 si tiene 3, de modo que
    las mismas celdas de visualizacion sirven para ambas variantes.
    """
    a = t.detach().cpu().numpy() if torch.is_tensor(t) else np.asarray(t)
    a = np.squeeze(a)
    if a.ndim == 3 and a.shape[0] in (1, 3):
        a = np.transpose(a, (1, 2, 0))
    return np.clip(a, 0, 1)


CMAP = "gray" if IN_CH == 1 else None

### 1.6. Datasets y DataLoaders para clasificacion

In [ ]:
class NoisyCIFARDataset(Dataset):
    def __init__(self, base_dataset, seed):
        self.base = base_dataset
        self.generator = torch.Generator()
        self.generator.manual_seed(seed)

        self.noisy_images = []
        self.labels = []

        for img, _ in self.base:
            noisy_img, label = apply_mixed_noise(img, generator=self.generator)
            self.noisy_images.append(noisy_img)
            self.labels.append(label)

        if len(self.noisy_images) != len(self.labels):
            raise Exception("Incompatible arrays")

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        return self.noisy_images[idx], self.labels[idx]


train_dataset = NoisyCIFARDataset(train_subset, seed=SEED)
val_dataset   = NoisyCIFARDataset(val_subset, seed=SEED + 1)
test_dataset  = NoisyCIFARDataset(X_test, seed=SEED + 2)

In [ ]:
train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=train_generator)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### 1.7. Inspeccion visual del ruido

In [ ]:
imgs, lbls = next(iter(train_loader))
names = {0: "gauss+speckle", 1: "gauss+S&P", 2: "S&P+speckle"}

fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
for i in range(8):
    axes[i].imshow(to_disp(imgs[i]), cmap=CMAP)
    axes[i].set_title(names[int(lbls[i])], fontsize=9)
    axes[i].axis("off")

plt.suptitle("CIFAR-100 con ruido mixto")
plt.tight_layout()
plt.show()

## 2. Clasificacion de ruido con HQNN hibrida (CNN + QNN)

### 2.1. Bloque cuantico: `EstimatorQNN`

Circuito de 4 qubits: `ZZFeatureMap` con entrelazamiento completo y una
repeticion, seguido de `RealAmplitudes` con entrelazamiento reverse-linear.
Los tres observables se leen directamente como logits de las tres clases
(`fc_final = Identity`).

In [ ]:
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit import QuantumCircuit
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

estimator = Estimator()

observables = [
    SparsePauliOp.from_list([("ZIII", 1.0), ("IZII", 1.0)]),
    SparsePauliOp.from_list([("ZZII", 1.0)]),
    SparsePauliOp.from_list([("IIZZ", 1.0)]),
]


def create_qnn():
    feature_map = zz_feature_map(4, entanglement="full", reps=1)
    ansatz = real_amplitudes(4, entanglement="reverse_linear", reps=1)

    qc = QuantumCircuit(4)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        input_gradients=True,
        estimator=estimator,
        observables=observables,
    )
    return qnn


qnn_clf = create_qnn()
qnn_clf.circuit.draw("mpl")

### 2.2. Arquitectura hibrida completa

In [ ]:
class Net(nn.Module):
    def __init__(self, qnn):
        super().__init__()
        self.conv1 = nn.Conv2d(IN_CH, 16, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, padding=2)
        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(CLF_FLAT, 128)
        self.fc2 = nn.Linear(128, 4)

        self.qnn = TorchConnector(qnn)
        self.fc_final = nn.Identity()

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)

        # .reshape() en vez de .view(): LIME genera tensores no contiguos.
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = torch.tanh(self.fc2(x)) * np.pi

        x = self.qnn(x)
        x = self.fc_final(x)
        return x


model_clf = Net(qnn_clf).to(device)
model_clf

### 2.3. Entrenamiento

In [ ]:
optimizer_clf = optim.AdamW(model_clf.parameters(), lr=1e-3)
criterion_clf = nn.CrossEntropyLoss()

epochs = 10
train_loss_list, val_loss_list = [], []
train_acc_list, val_acc_list = [], []

start_time = time.time()
patience = 3
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(epochs):

    model_clf.train()
    total_loss, correct, total = 0, 0, 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer_clf.zero_grad()
        output = model_clf(data)
        loss = criterion_clf(output, target)
        loss.backward()
        optimizer_clf.step()

        total_loss += loss.item() * data.size(0)
        _, predicted = torch.max(output, 1)
        correct += (predicted == target).sum().item()
        total += target.size(0)

    avg_train_loss = total_loss / total
    train_acc = correct / total
    train_loss_list.append(avg_train_loss)
    train_acc_list.append(train_acc)

    model_clf.eval()
    val_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model_clf(data)
            loss = criterion_clf(output, target)
            val_loss += loss.item() * data.size(0)
            _, predicted = torch.max(output, 1)
            correct += (predicted == target).sum().item()
            total += target.size(0)

    avg_val_loss = val_loss / total
    val_acc = correct / total
    val_loss_list.append(avg_val_loss)
    val_acc_list.append(val_acc)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = model_clf.state_dict()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping.")
            break

training_time = time.time() - start_time
model_clf.load_state_dict(best_model_state)

print(f"Tiempo de entrenamiento: {training_time:.2f} segundos")
print(f"Tiempo en minutos y segundos: {int(training_time // 60)} minutos y {int(training_time % 60)} segundos")

In [ ]:
plot_training_curves(train_loss_list, val_loss_list, train_acc_list, val_acc_list)

### 2.4. Evaluacion en test y matriz de confusion

In [ ]:
results_clf = evaluate_model(model_clf, test_loader, device)

print("Accuracy:", results_clf["accuracy"])
print(results_clf["report"])

plot_confusion_matrix(
    results_clf["confusion_matrix"],
    ["class 0", "class 1", "class 2"]
)

Compara esta accuracy con la del notebook gemelo `Classical_CNN_CVAE.ipynb` y, sobre todo,
con la obtenida por este mismo modelo sobre MNIST. Una caida sustancial en
CIFAR-100 seria coherente con la hipotesis de que parte del rendimiento en MNIST
proviene del atajo del fondo negro descrito en la cabecera.

## 3. Denoising de imagenes con ConvHQVAE

### 3.1. Dataset de denoising

In [ ]:
class DenoisingCIFARDataset(Dataset):
    def __init__(self, base_dataset, seed):
        self.base = base_dataset
        self.generator = torch.Generator()
        self.generator.manual_seed(seed)

        self.noisy_images = []
        self.clean_images = []

        for img, _ in self.base:
            noisy_img, _ = apply_mixed_noise(img, generator=self.generator)
            self.clean_images.append(img)
            self.noisy_images.append(noisy_img)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        return self.noisy_images[idx], self.clean_images[idx]


train_dataset_dn = DenoisingCIFARDataset(train_subset, seed=SEED)
val_dataset_dn   = DenoisingCIFARDataset(val_subset, seed=SEED + 1)
test_dataset_dn  = DenoisingCIFARDataset(X_test, seed=SEED + 2)

train_generator_dn = torch.Generator()
train_generator_dn.manual_seed(SEED)

train_loader_dn = DataLoader(train_dataset_dn, batch_size=16, shuffle=True, generator=train_generator_dn)
val_loader_dn   = DataLoader(val_dataset_dn, batch_size=16, shuffle=False)
test_loader_dn  = DataLoader(test_dataset_dn, batch_size=16, shuffle=False)

### 3.2. Funcion de perdida ($\beta$-VAE)

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=0.5):
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    total = bce + beta * kl
    return total, bce, kl

### 3.3. Bloque cuantico del espacio latente (6 qubits)

Circuito de 6 qubits con `ZZFeatureMap` (entrelazamiento completo) y
`RealAmplitudes` (entrelazamiento completo). Los seis observables forman una
correlacion en anillo entre qubits vecinos.

In [ ]:
observables_vae = [  # correlacion en anillo
    SparsePauliOp.from_list([("ZZIIII", 1.0)]),  # q0-q1
    SparsePauliOp.from_list([("IZZIII", 1.0)]),  # q1-q2
    SparsePauliOp.from_list([("IIZZII", 1.0)]),  # q2-q3
    SparsePauliOp.from_list([("IIIZZI", 1.0)]),  # q3-q4
    SparsePauliOp.from_list([("IIIIZZ", 1.0)]),  # q4-q5
    SparsePauliOp.from_list([("ZIIIIZ", 1.0)]),  # q5-q0
]


def create_qnn_vae():
    n_qubits = 6
    feature_map = zz_feature_map(n_qubits, reps=1, entanglement="full")
    ansatz = real_amplitudes(n_qubits, entanglement="full", reps=1)

    qc = QuantumCircuit(n_qubits)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        input_gradients=True,
        estimator=estimator,
        observables=observables_vae,
    )
    return qnn


qnn_vae = create_qnn_vae()
N_OBS = len(observables_vae)

### 3.4. Arquitectura del ConvHQVAE

In [ ]:
class ConvHQVAE(nn.Module):
    def __init__(self, latent_dim=12, qnn=None):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(IN_CH, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu = nn.Linear(ENC_FLAT, latent_dim)
        self.fc_logvar = nn.Linear(ENC_FLAT, latent_dim)

        self.qnn = TorchConnector(qnn) if qnn is not None else nn.Identity()

        n_qnn_inputs = qnn.num_inputs if qnn is not None else latent_dim
        self.to_qnn = (
            nn.Linear(latent_dim, n_qnn_inputs)
            if latent_dim != n_qnn_inputs else nn.Identity()
        )

        self.fc_decode = nn.Linear(latent_dim + N_OBS, ENC_FLAT)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, FEAT, FEAT)),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, IN_CH, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparametrize(self, mu, logvar):
        sigma = torch.exp(0.5 * logvar)
        return mu + sigma * torch.randn_like(sigma)

    def decode(self, z):
        z_qnn = self.qnn(self.to_qnn(z))
        z_combined = torch.cat([z, z_qnn], dim=1)
        return self.decoder(self.fc_decode(z_combined))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        return self.decode(z), mu, logvar

    def reconstruct(self, x):
        mu, logvar = self.encode(x)
        return self.decode(mu)


model_vae = ConvHQVAE(latent_dim=12, qnn=qnn_vae).to(device)
model_vae

### 3.5. Entrenamiento

In [ ]:
optimizer_vae = optim.AdamW(model_vae.parameters(), lr=1e-3)

EPOCHS = 15

vae_train_loss, vae_val_loss = [], []
vae_train_bce, vae_val_bce = [], []
vae_train_kl, vae_val_kl = [], []

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    model_vae.train()
    train_loss = train_bce = train_kl = 0
    n_train = 0

    for noisy, clean in train_loader_dn:
        noisy, clean = noisy.to(device), clean.to(device)

        optimizer_vae.zero_grad()
        recon, mu, logvar = model_vae(noisy)
        loss, bce, kl = vae_loss(recon, clean, mu, logvar)
        loss.backward()
        optimizer_vae.step()

        bs = noisy.size(0)
        train_loss += loss.item() * bs
        train_bce += bce.item() * bs
        train_kl += kl.item() * bs
        n_train += bs

    train_loss /= n_train
    train_bce /= n_train
    train_kl /= n_train

    model_vae.eval()
    val_loss = val_bce = val_kl = 0
    n_val = 0

    with torch.no_grad():
        for noisy, clean in val_loader_dn:
            noisy, clean = noisy.to(device), clean.to(device)
            recon, mu, logvar = model_vae(noisy)
            loss, bce, kl = vae_loss(recon, clean, mu, logvar)

            bs = noisy.size(0)
            val_loss += loss.item() * bs
            val_bce += bce.item() * bs
            val_kl += kl.item() * bs
            n_val += bs

    val_loss /= n_val
    val_bce /= n_val
    val_kl /= n_val

    vae_train_loss.append(train_loss); vae_val_loss.append(val_loss)
    vae_train_bce.append(train_bce); vae_val_bce.append(val_bce)
    vae_train_kl.append(train_kl); vae_val_kl.append(val_kl)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Train BCE: {train_bce:.4f} | Val BCE: {val_bce:.4f} | "
        f"Train KL: {train_kl:.4f} | Val KL: {val_kl:.4f}"
    )

training_time = time.time() - start_time
print(f"Tiempo de entrenamiento: {training_time:.2f} segundos")
print(f"Tiempo en minutos y segundos: {int(training_time // 60)} minutos y {int(training_time % 60)} segundos")

### 3.6. Reconstrucciones y metricas cuantitativas

In [ ]:
mostrar_reconstrucciones(model_vae, test_dataset_dn, n=8, device=device)

In [ ]:
from sklearn.metrics import mean_squared_error


def evaluate_vae(model, dataset, device):
    model.eval()
    mse_list, psnr_list = [], []

    with torch.no_grad():
        for noisy, clean in DataLoader(dataset, batch_size=32, shuffle=False):
            noisy, clean = noisy.to(device), clean.to(device)
            recon = model.reconstruct(noisy)

            mse = F.mse_loss(recon, clean, reduction='none')
            mse = mse.view(mse.size(0), -1).mean(dim=1)
            mse_list.extend(mse.cpu().numpy())

            psnr = 10 * np.log10(1.0 / (mse.cpu().numpy() + 1e-10))
            psnr_list.extend(psnr)

    return {"MSE": np.mean(mse_list), "PSNR": np.mean(psnr_list)}

In [ ]:
from skimage.metrics import structural_similarity as ssim


def ssim_dataset(model, loader, device):
    """SSIM medio sobre el loader. `channel_axis` se activa solo en RGB."""
    model.eval()
    vals = []

    with torch.no_grad():
        for noisy, clean in loader:
            noisy, clean = noisy.to(device), clean.to(device)
            recon = model.reconstruct(noisy)

            recon_np = recon.cpu().numpy()
            clean_np = clean.cpu().numpy()

            for i in range(recon_np.shape[0]):
                r, c = recon_np[i], clean_np[i]
                if IN_CH == 1:
                    vals.append(ssim(c[0], r[0], data_range=1.0))
                else:
                    vals.append(
                        ssim(
                            np.transpose(c, (1, 2, 0)),
                            np.transpose(r, (1, 2, 0)),
                            data_range=1.0,
                            channel_axis=-1,
                        )
                    )

    return float(np.mean(vals))

In [ ]:
vae_results = evaluate_vae(model_vae, test_dataset_dn, device)
vae_ssim = ssim_dataset(model_vae, test_loader_dn, device)

print("ConvHQVAE sobre CIFAR-100:")
print("  MSE :", round(float(vae_results["MSE"]), 6))
print("  PSNR:", round(float(vae_results["PSNR"]), 4), "dB")
print("  SSIM:", round(vae_ssim, 4))

## 4. Explicabilidad (XAI) del clasificador (HQNN hibrida (CNN + QNN))

Se aplica la misma bateria de tecnicas que en `Classical_CNN_CVAE.ipynb` sobre `model_clf`,
de modo que las figuras de ambos notebooks sean directamente superponibles.

In [ ]:
from captum.attr import (
    Saliency,
    IntegratedGradients,
    Occlusion,
    LayerGradCam,
    LayerAttribution,
)

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

### 4.1. Visualizacion de activaciones internas (`conv1`, `conv2`)

In [ ]:
activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach().cpu()
    return hook

hook1 = model_clf.conv1.register_forward_hook(get_activation('conv1'))
hook2 = model_clf.conv2.register_forward_hook(get_activation('conv2'))

In [ ]:
model_clf.eval()

images, labels = next(iter(test_loader))
image = images[0].unsqueeze(0).to(device)
label = labels[0].item()

with torch.no_grad():
    pred = model_clf(image).argmax(dim=1).item()

print(f"Clase real: {label}")
print(f"Clase predicha: {pred}")

_ = model_clf(image)  # forward pass para poblar `activations`

In [ ]:
act = activations['conv1'][0]

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(act[i], cmap='viridis')
    ax.set_title(f'Filtro {i}')
    ax.axis('off')

plt.suptitle("Activaciones conv1")
plt.tight_layout()
plt.show()

In [ ]:
act = activations['conv2'][0]

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(act[i], cmap='viridis')
    ax.set_title(f'Filtro {i}')
    ax.axis('off')

plt.suptitle("Activaciones conv2")
plt.tight_layout()
plt.show()

**Interpretacion**: `conv1` y `conv2` son estructuralmente identicas a las del
notebook gemelo, asi que cualquier diferencia observada aqui no puede atribuirse
a la naturaleza cuantica del modelo,
sino a la trayectoria de optimizacion. Esto convierte a esta seccion en un
control: si las activaciones tempranas son similares entre ambos notebooks, el
efecto del bloque final se concentra aguas abajo.

### 4.2. Saliency Maps

In [ ]:
saliency = Saliency(model_clf)

image.requires_grad = True
attribution = saliency.attribute(image, target=pred)

# En RGB se agrega el valor absoluto sobre los canales para obtener un mapa 2D,
# que es lo que se compara entre el notebook clasico y el hibrido.
attr = attribution.abs().squeeze(0)
attr = attr.max(dim=0).values if IN_CH > 1 else attr.squeeze(0)
attr = attr.cpu().detach().numpy()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(to_disp(image), cmap=CMAP)
axes[0].set_title("Imagen ruidosa")

axes[1].imshow(attr, cmap='hot')
axes[1].set_title("Saliency Map")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

**Interpretacion**: el mapa refleja las regiones cuya perturbacion alteraria mas
la prediccion. El gradiente atraviesa aqui la QNN completa (regla de la cadena a traves de `TorchConnector`), por lo que recoge la sensibilidad conjunta del extractor convolucional y del circuito variacional.
En CIFAR-100 conviene fijarse en si la atribucion se distribuye por toda la
imagen (coherente con un ruido que afecta a todos los pixeles) o se concentra en
regiones concretas.

### 4.3. Integrated Gradients

In [ ]:
ig = IntegratedGradients(model_clf)

attr_ig = ig.attribute(image, target=pred, n_steps=50).squeeze(0)
attr_ig = attr_ig.sum(dim=0) if IN_CH > 1 else attr_ig.squeeze(0)
attr_ig = attr_ig.cpu().detach().numpy()

In [ ]:
max_val = np.max(np.abs(attr_ig))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(to_disp(image), cmap=CMAP)
axes[0].set_title("Imagen ruidosa")

axes[1].imshow(attr_ig, cmap='coolwarm', vmin=-max_val, vmax=max_val)
axes[1].set_title("Integrated Gradients")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

**Interpretacion**: Integrated Gradients produce explicaciones mas suaves y
estables que Saliency. Al ser un metodo axiomatico (cumple *completeness* y
*sensitivity*), es uno de los candidatos mas solidos para comparar
cuantitativamente ambas ramas en la memoria.

### 4.4. Grad-CAM

In [ ]:
gradcam = LayerGradCam(model_clf, model_clf.conv2)
attr_gc = gradcam.attribute(image, target=pred)

attr_gc_up = LayerAttribution.interpolate(attr_gc, image.shape[2:])
heatmap = attr_gc_up.squeeze().cpu().detach().numpy()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(to_disp(image), cmap=CMAP)
axes[0].set_title("Imagen original")

axes[1].imshow(to_disp(image), cmap=CMAP)
axes[1].imshow(heatmap, cmap='jet', alpha=0.5)
axes[1].set_title("Grad-CAM")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

**Interpretacion**: Grad-CAM lee de `conv2`, es decir, informacion previa al
bloque final. Es por tanto el metodo menos sensible a la naturaleza de ese
bloque y el punto de comparacion mas estable entre los dos notebooks.

### 4.5. Occlusion Sensitivity

In [ ]:
occlusion = Occlusion(model_clf)

# La ventana cubre todos los canales a la vez (IN_CH), de modo que se ocluye la
# region espacial completa y no solo un canal.
attr_occ = occlusion.attribute(
    image,
    strides=(IN_CH, 4, 4),
    target=pred,
    sliding_window_shapes=(IN_CH, 4, 4),
    baselines=0,
)

attr_occ = attr_occ.squeeze(0)
attr_occ = attr_occ.mean(dim=0) if IN_CH > 1 else attr_occ.squeeze(0)
attr_occ = attr_occ.cpu().detach().numpy()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(to_disp(image), cmap=CMAP)
axes[0].set_title("Imagen original")

axes[1].imshow(attr_occ, cmap='jet')
axes[1].set_title("Occlusion Sensitivity")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

Ventana de $4\times4$ y stride $4\times4$ sobre todos los canales: la imagen de
$\mathrm{IMG\_SIZE}\times\mathrm{IMG\_SIZE}$ queda dividida en una rejilla
discreta de bloques.
**Coste computacional**: cada bloque exige una inferencia forward completa, incluida la simulacion del circuito cuantico, por lo que este paso es sensiblemente mas lento que en la rama clasica.

### 4.6. Espacio latente pre-bloque final (4 dimensiones)

In [ ]:
model_clf.eval()

latent_vectors, latent_labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)

        x1 = F.relu(model_clf.conv1(x))
        x1 = model_clf.pool(x1)
        x1 = F.relu(model_clf.conv2(x1))
        x1 = model_clf.pool(x1)
        x1 = x1.reshape(x1.size(0), -1)
        x1 = F.relu(model_clf.fc1(x1))

        latent = torch.tanh(model_clf.fc2(x1)) * np.pi

        latent_vectors.append(latent.cpu().numpy())
        latent_labels.append(y.numpy())

latent_vectors = np.concatenate(latent_vectors)
latent_labels = np.concatenate(latent_labels)
print(latent_vectors.shape)

In [ ]:
tsne = TSNE(n_components=2, perplexity=20, random_state=42)
latent_2d = tsne.fit_transform(latent_vectors)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(latent_2d[:, 0], latent_2d[:, 1], c=latent_labels, cmap='tab10')
plt.colorbar(scatter, label="clase de ruido")
plt.title("Espacio latente 4D (pre-QNN) proyectado con t-SNE — CIFAR-100")
plt.show()

### 4.6-bis. Embedding post-QNN (3 dimensiones)

In [ ]:
model_clf.eval()

post_qnn_vectors = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)

        x1 = F.relu(model_clf.conv1(x))
        x1 = model_clf.pool(x1)
        x1 = F.relu(model_clf.conv2(x1))
        x1 = model_clf.pool(x1)
        x1 = x1.reshape(x1.size(0), -1)
        x1 = F.relu(model_clf.fc1(x1))

        pre_qnn = torch.tanh(model_clf.fc2(x1)) * np.pi
        post_qnn = model_clf.qnn(pre_qnn)

        post_qnn_vectors.append(post_qnn.cpu().numpy())

post_qnn_vectors = np.concatenate(post_qnn_vectors)

tsne_post = TSNE(n_components=2, perplexity=20, random_state=42)
post_qnn_2d = tsne_post.fit_transform(post_qnn_vectors)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(post_qnn_2d[:, 0], post_qnn_2d[:, 1], c=latent_labels, cmap='tab10')
plt.colorbar(scatter, label="clase de ruido")
plt.title("Embedding post-QNN (3D, valores esperados) proyectado con t-SNE — CIFAR-100")
plt.show()

**Interpretacion**: comparar esta proyeccion con la de la celda anterior es la
evidencia mas directa disponible sobre el papel del circuito. Si ambas muestran
una separabilidad similar, la QNN aplica una transformacion aproximadamente
conservadora sobre una representacion que la CNN ya habia hecho casi separable;
si la separacion post-QNN es visiblemente mayor, hay indicio de que el circuito
aporta valor discriminativo adicional.

### 4.7. Analisis de errores

In [ ]:
model_clf.eval()
wrong_images, wrong_preds, wrong_labels = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model_clf(x)
        preds = out.argmax(dim=1).cpu()

        wrong = preds != y
        wrong_images.extend(x.cpu()[wrong])
        wrong_preds.extend(preds[wrong])
        wrong_labels.extend(y[wrong])

print(f"Errores de clasificacion: {len(wrong_images)} de {len(test_dataset)}")

In [ ]:
n = min(5, len(wrong_images))

if n == 0:
    print("No se encontraron errores de clasificacion.")
else:
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    if n == 1:
        axes = [axes]

    for i in range(n):
        axes[i].imshow(to_disp(wrong_images[i]), cmap=CMAP)
        axes[i].set_title(f"Real:{wrong_labels[i]}\nPred:{wrong_preds[i]}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

En MNIST, con un test de 100 muestras y un problema muy separable, era habitual
no encontrar errores. En CIFAR-100 es esperable que si aparezcan, lo que hace
esta seccion mas informativa: conviene observar si los fallos se concentran en
alguna clase concreta.

### 4.8. LIME para explicacion local

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries, slic

In [ ]:
def predict_lime(images):
    """LIME siempre entrega lotes RGB (N, H, W, 3).

    Si el modelo espera 1 canal se promedian los canales para recuperar la
    escala de grises; si espera 3 se usan tal cual.
    """
    model_clf.eval()

    if IN_CH == 1:
        arr = np.mean(images, axis=-1, keepdims=True).astype(np.float32)
    else:
        arr = images.astype(np.float32)

    tensor = torch.tensor(arr).permute(0, 3, 1, 2).to(device)

    with torch.no_grad():
        outputs = model_clf(tensor)
        probs = torch.softmax(outputs, dim=1)

    return probs.cpu().numpy()

In [ ]:
model_clf.eval()

image_test = label_real = label_pred = None

for noisy_img, lbl in test_dataset:
    x = noisy_img.unsqueeze(0).to(device)

    with torch.no_grad():
        pred_lime = model_clf(x).argmax(1).item()

    if pred_lime == lbl:
        image_test = noisy_img
        label_real = lbl
        label_pred = pred_lime
        break

print("Instancia elegida | real:", label_real, "| pred:", label_pred)

In [ ]:
img_np = to_disp(image_test)

# LIME requiere entrada RGB: si la imagen es de 1 canal se replica.
img_rgb = np.stack([img_np] * 3, axis=-1) if img_np.ndim == 2 else img_np

explainer = lime_image.LimeImageExplainer()

segmenter = lambda x: slic(x, n_segments=20, compactness=5, start_label=1)

explanation = explainer.explain_instance(
    img_rgb.astype(np.double),
    predict_lime,
    top_labels=3,
    hide_color=0,
    num_samples=500,
    segmentation_fn=segmenter,
)

In [ ]:
temp, mask = explanation.get_image_and_mask(
    label_pred,
    positive_only=True,
    num_features=5,
    hide_rest=False,
)

plt.figure(figsize=(5, 5))
plt.imshow(mark_boundaries(temp, mask))
plt.title(f"LIME\nReal: {label_real} | Pred: {label_pred}")
plt.axis('off')
plt.show()

**Interpretacion**: al ser un metodo de caja negra, LIME no distingue entre las
capas internas del modelo; solo observa la relacion entrada-salida.
**Coste computacional**: las 500 perturbaciones requieren 500 inferencias, cada una con su simulacion de circuito, por lo que el tiempo de ejecucion es notablemente superior al de la rama clasica.

### 4.9. SHAP (GradientExplainer)

In [ ]:
import shap

model_clf.eval()

background = []
for i in range(50):
    img, _ = train_dataset[i]
    background.append(img.numpy())
background = np.array(background)

print("background:", background.shape)

In [ ]:
image_test_shap, label_real_shap = test_dataset[0]
image_batch = image_test_shap.unsqueeze(0).numpy()

explainer = shap.GradientExplainer(
    model_clf,
    torch.tensor(background, dtype=torch.float32).to(device),
)

shap_values = explainer.shap_values(
    torch.tensor(image_batch, dtype=torch.float32).to(device)
)

print(type(shap_values))
if isinstance(shap_values, list):
    print(len(shap_values), shap_values[0].shape)
else:
    print(shap_values.shape)

In [ ]:
# El formato de salida de shap varia segun la version: puede ser una lista con
# un array por clase, o un unico array con la clase en el ultimo eje.
if isinstance(shap_values, list):
    sv = shap_values[label_real_shap][0]          # (C, H, W)
else:
    sv = shap_values[0, ..., label_real_shap]     # (C, H, W)

sv = np.asarray(sv)
map_shap = sv.sum(axis=0) if sv.ndim == 3 else sv

In [ ]:
max_val = np.max(np.abs(map_shap))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(to_disp(image_test_shap), cmap=CMAP)
axes[0].set_title(f"Original\nClase: {label_real_shap}")

im = axes[1].imshow(map_shap, cmap='coolwarm', vmin=-max_val, vmax=max_val)
axes[1].set_title("SHAP (GradientExplainer)")

for ax in axes:
    ax.axis('off')

plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

**Interpretacion**: las regiones rojas favorecen la clase predicha y las azules
la penalizan. *Hipotesis a contrastar con `Classical_CNN_CVAE.ipynb`*: cabe esperar que la
CNN clasica produzca mapas de relevancia mas suaves y espacialmente
distribuidos, mientras que la HQNN hibrida podria concentrar la relevancia en
regiones mas discretas como consecuencia de la compresion que impone el paso por
un circuito de solo 4 qubits. Esta hipotesis debe verificarse comparando las
figuras de ambos notebooks, no darse por supuesta.

## 5. Explicabilidad (XAI) del modelo de denoising (ConvHQVAE)

A diferencia del clasificador, aqui no se explica una clase sino una
reconstruccion: el foco pasa a ser la geometria del espacio latente y el
comportamiento del decoder.

### 5.1. Reconstrucciones y analisis cualitativo

In [ ]:
model_vae.eval()

fig, axes = plt.subplots(8, 3, figsize=(9, 20))

with torch.no_grad():
    for i in range(8):
        noisy, clean = test_dataset_dn[i]
        recon, mu, logvar = model_vae(noisy.unsqueeze(0).to(device))

        axes[i, 0].imshow(to_disp(noisy), cmap=CMAP); axes[i, 0].set_title("Noisy")
        axes[i, 1].imshow(to_disp(clean), cmap=CMAP); axes[i, 1].set_title("Clean")
        axes[i, 2].imshow(to_disp(recon), cmap=CMAP); axes[i, 2].set_title("Reconstruction")

        for j in range(3):
            axes[i, j].axis('off')

plt.tight_layout()
plt.show()

**Analisis esperado**: con 15 epocas y 300 muestras de entrenamiento sobre
imagenes naturales, las reconstrucciones seran considerablemente mas borrosas
que en MNIST. Lo relevante para la memoria no es la calidad absoluta sino la
comparacion con el notebook gemelo bajo identico presupuesto.

### 5.2. Organizacion del espacio latente ($\mu$) con PCA

In [ ]:
latent_vectors_vae = []

model_vae.eval()
with torch.no_grad():
    for noisy, clean in test_loader_dn:
        noisy = noisy.to(device)
        mu, logvar = model_vae.encode(noisy)
        latent_vectors_vae.append(mu.cpu())

latent_vectors_vae = torch.cat(latent_vectors_vae).numpy()
print(latent_vectors_vae.shape)

In [ ]:
pca = PCA(n_components=2)
latent_2d_vae = pca.fit_transform(latent_vectors_vae)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(
    latent_2d_vae[:, 0], latent_2d_vae[:, 1],
    c=np.arange(len(latent_2d_vae)), cmap='viridis', s=25,
)
plt.colorbar(scatter)
plt.title("PCA del espacio latente ($\\mu$) del ConvHQVAE — CIFAR-100")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

**Nota importante**: `mu` es la salida del encoder **clasico** (`fc_mu`), previa
a cualquier intervencion cuantica.
Por tanto esta proyeccion es directamente comparable entre ambos notebooks: mide
como ha organizado el encoder el espacio latente, no lo que hace el bloque
posterior.

### 5.3. t-SNE del espacio latente

In [ ]:
tsne_vae = TSNE(n_components=2, perplexity=15, random_state=42)
latent_tsne = tsne_vae.fit_transform(latent_vectors_vae)

plt.figure(figsize=(7, 6))
plt.scatter(latent_tsne[:, 0], latent_tsne[:, 1], s=20)
plt.title("t-SNE del espacio latente — CIFAR-100")
plt.show()

**Interpretacion**: t-SNE revela relaciones no lineales que PCA, al ser una
proyeccion lineal, no puede capturar.

### 5.4. Que regiones afectan mas a la reconstruccion — Saliency del ConvHQVAE

In [ ]:
sample_noisy, sample_clean = test_dataset_dn[0]

input_img = sample_noisy.unsqueeze(0).to(device)
input_img.requires_grad = True

recon, _, _ = model_vae(input_img)
loss = F.mse_loss(recon, input_img)
loss.backward()

grad = input_img.grad.abs().squeeze(0)
saliency_vae = (grad.max(dim=0).values if IN_CH > 1 else grad.squeeze(0)).cpu().numpy()

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(to_disp(sample_noisy), cmap=CMAP)
plt.title("Entrada ruidosa")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(saliency_vae, cmap='hot')
plt.title("Saliency del ConvHQVAE")
plt.axis('off')

plt.show()

**Interpretacion**: el gradiente de la perdida de reconstruccion respecto a la
entrada atraviesa tambien aqui el circuito cuantico del cuello de botella.
En imagenes naturales cabe esperar una atribucion mas repartida que en MNIST,
donde se concentraba en el trazo del digito.

### 5.5. Que partes se pierden — Error residual

In [ ]:
sample_noisy, sample_clean = test_dataset_dn[0]

model_vae.eval()
with torch.no_grad():
    recon, _, _ = model_vae(sample_noisy.unsqueeze(0).to(device))

recon_d = to_disp(recon)
clean_d = to_disp(sample_clean)

error_map = np.abs(clean_d - recon_d)
if error_map.ndim == 3:
    error_map = error_map.mean(axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

axes[0].imshow(clean_d, cmap=CMAP); axes[0].set_title("Original"); axes[0].axis('off')
axes[1].imshow(recon_d, cmap=CMAP); axes[1].set_title("Reconstruccion"); axes[1].axis('off')
axes[2].imshow(error_map, cmap='hot'); axes[2].set_title("Error residual"); axes[2].axis('off')

plt.tight_layout()
plt.show()

**Interpretacion**: el error residual $|x - \hat{x}|$ suele concentrarse en
bordes y texturas de alta frecuencia, que son precisamente las que un latente de
dimension 12 no puede representar.

### 5.6. Que imagina el decoder — Traversals latentes

In [ ]:
latent_dim = 12

z = torch.zeros((1, latent_dim)).to(device)
values = torch.linspace(-3, 3, steps=8)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))

for dim in range(2):
    for i, val in enumerate(values):
        z_mod = z.clone()
        z_mod[0, dim] = val
        with torch.no_grad():
            recon = model_vae.decode(z_mod)
        axes[dim, i].imshow(to_disp(recon), cmap=CMAP)
        axes[dim, i].axis('off')
        if dim == 0:
            axes[dim, i].set_title(f"{val:.1f}")
    axes[dim, 0].set_ylabel(f"dim {dim}")

plt.suptitle("Traversals latentes (dimensiones 0 y 1) — CIFAR-100")
plt.tight_layout()
plt.show()

**Interpretacion**: al recorrer cada dimension latente manteniendo el resto en 0,
`decode()` muestra que factor de variacion ha capturado esa coordenada.
Notese que el recorrido atraviesa tambien el circuito cuantico, de modo que lo que se visualiza es el efecto combinado del latente y de la QNN.

## 6. Conclusiones y guia de comparacion con el notebook gemelo

Para que la comparacion con `Classical_CNN_CVAE.ipynb` sea util, conviene contrastar
seccion por seccion:

| Seccion | Que comparar |
|---|---|
| 2.3 / 2.4 | Curvas de entrenamiento, accuracy final y matriz de confusion |
| 3.5 / 3.6 | Curvas de perdida y valores de MSE, PSNR y SSIM |
| 4.1 | Activaciones tempranas (control: deberian parecerse) |
| 4.2 - 4.5 | Mapas de atribucion: dispersion, concentracion y coincidencia |
| 4.6 / 4.6-bis | Separabilidad del espacio latente antes y despues del circuito |
| 4.5 / 4.8 | Coste computacional relativo de los metodos por perturbacion |
| 5.2 / 5.3 | Geometria del espacio $\mu$ |
| 5.6 | Interpretabilidad de los traversals |

Y, de forma transversal, el contraste con los resultados equivalentes sobre
MNIST: si el modelo hibrido mantiene su ventaja o desventaja relativa al pasar a
imagenes naturales, la conclusion obtenida en MNIST gana solidez; si cambia,
significa que dependia de las particularidades del dataset.

### Exportacion a HTML

In [ ]:
import sys
import subprocess
from pathlib import Path

notebook = Path("HQ_CNN_CVAE.ipynb").resolve()
output_dir = notebook.parent

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "nbconvert",
        "--to", "html",
        str(notebook),
        "--output-dir", str(output_dir),
    ],
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---")
print(result.stdout)
print("\n--- STDERR ---")
print(result.stderr)

if result.returncode == 0:
    print("\nHTML generado correctamente:")
    print(output_dir / "HQ_CNN_CVAE.html")